# 11.02 · Gateways, routing & caching

**Chapter:** LLMOps, Observability & Safety · **Milestone:** M5

## Goals
- [ ] Fallbacks and retries across providers with the LiteLLM `Router`
- [ ] Know the gateway options: LiteLLM proxy, OpenRouter, Portkey, Cloudflare AI Gateway, cloud-native gateways
- [ ] Exact and **semantic** caching; budgets and per-key limits
- [ ] Cost-aware model routing

## Resources
- [LiteLLM – Router (fallbacks, retries, load balancing)](https://docs.litellm.ai/docs/routing) · [Proxy server](https://docs.litellm.ai/docs/simple_proxy) — free/OSS
- [OpenRouter – Model fallbacks & provider routing](https://openrouter.ai/docs/features/model-routing) — free tier
- [RouteLLM (learned routers)](https://github.com/lm-sys/RouteLLM) — free

A **gateway** sits between your app and providers: one API, many models; keys, budgets and rate limits in one place; fallbacks when a
provider is down; logging and caching. In-process (a library) or as a proxy server your whole org uses.

In [ ]:
import os
import time

import numpy as np

from aieng import rag
from aieng.llm import client

c, MODEL = client()
LOCAL = c.base_url.host == "localhost"
EXTRA = {"reasoning_effort": "none"} if LOCAL else {}

## 1. Fallbacks with the LiteLLM Router
Simulate an outage: "primary" points at a port where nothing listens. The router retries, then falls back to "backup".

In [ ]:
import litellm
from litellm import Router

litellm.suppress_debug_info = True
router = Router(
    model_list=[
        {"model_name": "primary", "litellm_params": {"model": f"openai/{MODEL}", "api_base": "http://localhost:9/v1", "api_key": "x", "timeout": 2}},
        {"model_name": "backup", "litellm_params": {"model": f"openai/{MODEL}", "api_base": str(c.base_url), "api_key": c.api_key, **EXTRA}},
    ],
    fallbacks=[{"primary": ["backup"]}],
    num_retries=1,
)
start = time.perf_counter()
r = router.completion(model="primary", messages=[{"role": "user", "content": "Say 'ok'."}])
print(f"answered by {r._hidden_params.get('api_base')} in {time.perf_counter() - start:.1f}s: {r.choices[0].message.content!r}")

Real configs list several deployments per model name (load balancing across keys/regions/providers), `fallbacks` for outages,
`context_window_fallbacks` for prompts too long for one model, and `content_policy_fallbacks`.

## 2. Hosted routing: OpenRouter
One key, hundreds of models; `models=[...]` gives server-side fallbacks. Runs if `OPENROUTER_API_KEY` is set.

In [ ]:
if os.getenv("OPENROUTER_API_KEY"):
    orc, or_model = client("openrouter")
    try:
        r = orc.chat.completions.create(model=or_model, messages=[{"role": "user", "content": "Say 'ok'."}], max_tokens=20,
                                        extra_body={"models": [or_model, "meta-llama/llama-3.3-70b-instruct:free"]})
        print("OpenRouter served:", r.model, "->", r.choices[0].message.content)
    except Exception as e:  # free models get rate-limited often
        print("OpenRouter error:", type(e).__name__, str(e)[:150])
else:
    print("OPENROUTER_API_KEY not set — skipping")

## 3. LiteLLM as a proxy server (for a team)
Same router, as an OpenAI-compatible server with virtual keys, budgets, rate limits and a spend dashboard (needs a Postgres DB for keys/spend):
```yaml
# litellm_config.yaml
model_list:
  - model_name: fast
    litellm_params: {model: groq/llama-3.3-70b-versatile, api_key: os.environ/GROQ_API_KEY}
  - model_name: fast
    litellm_params: {model: gemini/gemini-flash-lite-latest, api_key: os.environ/GEMINI_API_KEY}   # same name = load balanced
  - model_name: local
    litellm_params: {model: ollama_chat/qwen3:1.7b, api_base: http://localhost:11434}
router_settings: {fallbacks: [{fast: [local]}], num_retries: 2}
litellm_settings: {cache: true}
```
`uvx 'litellm[proxy]' --config litellm_config.yaml` → apps use `OpenAI(base_url="http://localhost:4000", api_key="sk-<virtual key>")`.

## 4. Caching
**Exact cache**: same request → stored answer (05.05). **Semantic cache**: *similar* request → stored answer, via embedding similarity.
Huge savings for FAQ-style traffic; dangerous when small wording changes matter ("cancel my order" ≠ "don't cancel my order").

In [ ]:
class SemanticCache:
    def __init__(self, threshold=0.92):
        self.threshold, self.keys, self.values = threshold, [], []

    def get(self, prompt):
        if not self.keys:
            return None, 0.0
        sims = np.stack(self.keys) @ rag.embed([prompt])[0]
        i = int(np.argmax(sims))
        return (self.values[i], float(sims[i])) if sims[i] >= self.threshold else (None, float(sims[i]))

    def put(self, prompt, value):
        self.keys.append(rag.embed([prompt])[0]); self.values.append(value)


cache = SemanticCache()


def cached_chat(prompt):
    hit, sim = cache.get(prompt)
    if hit is not None:
        return hit, f"HIT (sim {sim:.2f})"
    answer = c.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], max_tokens=80, **EXTRA).choices[0].message.content
    cache.put(prompt, answer)
    return answer, f"miss (best sim {sim:.2f})"


for p in ["What are your opening hours?", "When are you open?", "What time do you open on weekdays?",
          "How do I cancel my order?", "How do I NOT cancel my order?"]:
    ans, status = cached_chat(p)
    print(f"{status:18} {p!r}")

Look at the last pair: if it was a HIT, the cache would return cancellation instructions to someone asking the opposite. Tune the threshold on
real pairs, scope caches per user/tenant where answers are personal, and never semantic-cache actions.
Libraries: GPTCache, Redis LangCache / RedisVL, LiteLLM's `cache` with `type="redis-semantic"`.

## 5. Budgets
A per-key/per-user spend guard is a few lines; gateways do it for teams.

In [ ]:
class Budget:
    def __init__(self, usd_limit, price_in=0.10, price_out=0.40):  # $/1M tokens, illustrative
        self.limit, self.spent, self.p_in, self.p_out = usd_limit, 0.0, price_in, price_out

    def charge(self, usage):
        self.spent += (usage.prompt_tokens * self.p_in + usage.completion_tokens * self.p_out) / 1e6
        if self.spent > self.limit:
            raise RuntimeError(f"budget exceeded: ${self.spent:.6f} > ${self.limit}")


b = Budget(usd_limit=0.0001)
try:
    for i in range(20):
        r = c.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Write a haiku about caching."}], max_tokens=60, **EXTRA)
        b.charge(r.usage)
except RuntimeError as e:
    print(f"stopped after {i + 1} calls:", e)

## 6. Cost-aware routing
Send easy requests to a cheap/fast model and hard ones to a strong model. Routers: rules (length, keywords), a classifier LLM (09.03),
or learned routers (RouteLLM, NotDiamond, Martian). Measure with evals: routing that saves 70% cost but fails 5% more tasks may not be a win.

## Exercises
1. Build a Router with `fast` = Groq and Gemini (load-balanced, both free tiers) and `local` = Ollama fallback. Kill your network and watch it fall back.
2. Calibrate the semantic-cache threshold: write 20 paraphrase pairs and 20 "similar but different meaning" pairs. Plot similarity distributions; pick a threshold.
3. Run the LiteLLM proxy with the config above and point `aieng.llm` at it (`base_url="http://localhost:4000"`).

In [ ]:
# your code here